In [0]:
import os
import json
import urllib.request
import urllib.parse
from datetime import date, timedelta
from delta.tables import DeltaTable
from pyspark.sql import functions as F


In [0]:
from pathlib import Path
import sys

from importlib.machinery import SourceFileLoader
from importlib.util import module_from_spec, spec_from_loader


arquivo_audit = Path.cwd() / "audit.py"

print(f"Arquivo: {arquivo_audit}")
print(f"É arquivo: {arquivo_audit.is_file()}")

if not arquivo_audit.is_file():
    raise FileNotFoundError(
        f"Arquivo audit.py não encontrado em: {arquivo_audit}"
    )

loader = SourceFileLoader(
    "audit",
    str(arquivo_audit),
)

spec = spec_from_loader(
    loader.name,
    loader,
)

audit = module_from_spec(spec)

# Registra o módulo para permitir futuros imports
sys.modules["audit"] = audit

loader.exec_module(audit)

from audit import (
    iniciar_auditoria,
    finalizar_auditoria,
    obter_metricas_delta,
)

print("audit.py carregado com sucesso")

In [0]:
from audit import (
    iniciar_auditoria,
    finalizar_auditoria,
    obter_metricas_delta)

In [0]:
from audit import TABELA_AUDITORIA

print(TABELA_AUDITORIA)

In [0]:
# Valores padrão permitem testar o notebook manualmente
dbutils.widgets.text("run_id", "MANUAL")
dbutils.widgets.text("job_name", "job_cambio_manual")
dbutils.widgets.text("task_name", "bronze_manual")

run_id = dbutils.widgets.get("run_id")
job_name = dbutils.widgets.get("job_name")
task_name = dbutils.widgets.get("task_name")

print(f"run_id: {run_id}")
print(f"job_name: {job_name}")
print(f"task_name: {task_name}")

In [0]:
from pyspark.sql import functions as F

catalogo = "databricks_cata_managed"
volume_landing = "cambio_ptax_raw_files"

# Janela móvel curta para carga incremental
data_fim = date.today()
data_inicio = data_fim 

data_inicio_api = data_inicio.strftime("%m-%d-%Y")
data_fim_api = data_fim.strftime("%m-%d-%Y")

data_inicio_ref = data_inicio.strftime("%Y-%m-%d")
data_fim_ref = data_fim.strftime("%Y-%m-%d")

batch_id = f"{data_inicio_ref}_{data_fim_ref}".replace("-", "")

tabela_bronze = f"{catalogo}.bronze.cambio_ptax_raw"
landing_dir = f"/Volumes/{catalogo}/landing/{volume_landing}/batch_{batch_id}"

tabela_bronze = f"{catalogo}.bronze.cambio_ptax_raw"

print(f"Lendo landing: {landing_dir}")
print(f"Salvando bronze: {tabela_bronze}")


In [0]:
df_bronze = (
    spark.read
    .option("multiLine", "true")
    .json(f"{landing_dir}/*.json")
    .withColumn("_arquivo_lido", F.col("_metadata.file_path"))
    .withColumn("_data_ingestao", F.current_timestamp())
    .withColumn("_batch_processamento", F.lit(batch_id))
    .withColumn("_camada", F.lit("bronze"))
    .withColumn("_origem", F.lit("API PTAX - Banco Central"))
)

display(df_bronze)

In [0]:

if not spark.catalog.tableExists(tabela_bronze):
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(tabela_bronze)
    )
else:
    delta_bronze = DeltaTable.forName(spark, tabela_bronze)

    (
        delta_bronze.alias("t")
        .merge(
            df_bronze.alias("s"),
            """
            t.batch_id = s.batch_id
            AND t.moeda = s.moeda
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )